One important thing to do before we start is to get an idea of how many files we will have to deal with. Depending on the number, it could influence our strategy of what we look at, as well as help inform us about the amount of storage we will use etc.

The files we are interested in are visibility files of our 41 stars that contain at least 1 hour of observing.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.table import Column
import tqdm
from astropy.table import join
from astroquery.simbad import Simbad
from ipyaladin import Aladin
from astroquery.casda import Casda
from astropy.coordinates import SkyCoord
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
import anywidget
import ipywidgets
import ipyaladin

In [2]:
OPAL_USER = 'jfer9940@uni.sydney.edu.au'
Casda.login(username=OPAL_USER, store_password=True)

INFO: Authenticating jfer9940@uni.sydney.edu.au on CASDA ... [astroquery.casda.core]
INFO: Authentication successful! [astroquery.casda.core]


The first step is to create a list of coordinates for our sources so that we can query CASDA, since CASDA doesn't accept source names in its searches.

Note there were 4 missing sources on first pass. When we investigate a little bit, we see they are in the SRSC, they just have an extra space in their main_id name that we need to account for, so let's edit these cells first before we do our coordinate match.

In [3]:
solar_like_stars = Table.read('data/simbad_classifications.csv')

In [4]:
solar_like_stars

main_id,sp_type,otype,notes
str19,str8,str3,str43
HD 175897,G5V,Ro*,--
HD 162186,G3IV(e),Ro*,--
BD+21 20A,--,*,solar type dwarf (Nascimbeni 2016)
BD+66 1664,G5,SB*,--
HD 217344,G4V,RS*,--
BD+09 4984B,F2IV/G5V,SB*,"listed as A2 (Anderson 2012, Raghavan 2010)"
V* V369 Gem,G5,RS*,--
HD 146696,G0,SB*,--
HD 156097,G5V,Er*,--


In [5]:
srsc_table = Table.read('data/2026.01.16_12_07_51_SRSC_Static.csv')

In [6]:
solar_like_simbad_ids = solar_like_stars['main_id'].tolist()

In [7]:
# Correct some of the names that were slightly different (missing spaces etc)
solar_like_simbad_ids[28] = 'BD+44 3670'
solar_like_simbad_ids[33] = 'HD 22213'
solar_like_simbad_ids[39] = 'HD 23524'
solar_like_simbad_ids[29] = 'V* V815 Her'

In [8]:
mask = np.isin(srsc_table['Simbad_ID'], solar_like_simbad_ids)
solar_like_coords = srsc_table[mask][['Simbad_ID', 'Archival_ra', 'Archival_dec']]

In [9]:
solar_like_coords

Simbad_ID,Archival_ra,Archival_dec
str31,float64,float64
HD 102982,177.787504285457,-51.876071495961
V* V376 Cep,329.577295376914,82.870902021925
V* V815 Her,272.06734740577,29.6910259723
HD 165590A,271.457229,21.44599
HD 82533,142.67769359508,-53.521472152447
HD 217344,345.11766870906,-33.745748964489
HD 8558,20.839329303534,-57.480907517274
HD 200244,315.742708658278,-22.933914165698
HD 162186,267.778490987213,-32.307805803789


Now that we have a list of our sources and their coordinates, we can do a CASDA query on those coordinates to find out what observations there are for each of these sources. Let's start with a test source and go from there once we know the layout.

In [10]:
ra = solar_like_coords[0]['Archival_ra']
dec = solar_like_coords[0]['Archival_dec']
coords = SkyCoord(ra = ra*u.deg, dec = dec*u.deg, frame='icrs')
ra

np.float64(177.787504285457)

In [11]:
result_table = Casda.query_region(coords, radius="5 arcmin")
public_data = Casda.filter_out_unreleased(result_table)

min_exptime = 60*60 # 1 Hour minimum time
dataproduct_type = 'visibility' # We want visibility data

mask = (public_data['dataproduct_type'] == dataproduct_type) & (public_data['t_exptime'] >= min_exptime)
results = public_data[mask]
results

# # n_obs = len(np.unique(results['obs_id']))
# obs_names = list(np.unique(results['obs_collection']).tolist())
# obs_names

distance,dataproduct_type,calib_level,obs_collection,obs_id,obs_publisher_did,access_url,access_format,access_estsize,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,s_xel1,s_xel2,t_min,t_max,t_exptime,t_resolution,t_xel,em_min,em_max,em_res_power,em_xel,o_ucd,pol_states,pol_xel,facility_name,instrument_name,dataproduct_subtype,em_ucd,em_unit,em_resolution,s_resolution_min,s_resolution_max,s_ucd,s_unit,obs_release_date,quality_level,thumbnail_id,filename
deg,,,,,,,,kbyte,,deg,deg,deg,,arcsec,,,d,d,s,s,,m,m,,,,,,,,,,,m,arcsec,arcsec,,,,,,
float64,str255,int32,str255,str255,str255,str2000,str255,int64,str255,float64,float64,float64,str200,float64,int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,int64,str255,str255,int64,str255,str255,str255,str255,str255,float64,float64,float64,str255,str255,str24,str15,int64,str1000
0.24667217102869798,visibility,1,FLASH,ASKAP-76312,scan-706397-3654160,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=scan-706397-3654160,application/x-votable+xml;content=datalink,6715900,FLASH_141,177.9233916151707,-51.644181117691396,1.82165,POLYGON ICRS 178.83421661517067 -51.644181117691396 178.83241931009135 -51.586989942741106 178.82703448799543 -51.530024475230185 178.81808340031566 -51.473509531833855 178.80560137290567 -51.41766815,--,--,--,60916.33284722222,60916.41625,7206.000000354834,--,1,0.29964540971199255,0.4207668124630408,--,288,phys.polarisation,/XX/XY/YX/YY/,4,ASKAP,,,em.wl,m,0.0004205604262189176,--,--,pos.eq,deg,2025-10-24T06:16:22.548Z,UNCERTAIN,--,scienceData.FLASH_141.SB76312.FLASH_141.beam10_averaged_cal.leakage.ms.tar
0.38012912205564936,visibility,1,WALLABY,ASKAP-72938,scan-642963-3527042,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=scan-642963-3527042,application/x-votable+xml;content=datalink,15749020,WALLABY_1142-53B,177.17178557951806,-51.87909381369503,1.11717,POLYGON ICRS 177.73037057951805 -51.87909381369503 177.72926833961716 -51.84401997134375 177.7259659699518 -51.80908454942451 177.72047650346633 -51.77442542208716 177.7128216045771 -51.74017937307305,--,--,--,60781.48490740741,60781.81841435185,28814.99999994412,--,1,0.2143032906152736,0.2150177436578653,--,251,phys.polarisation,/XX/XY/YX/YY/,4,ASKAP,,,em.wl,m,2.8464264645088117e-06,--,--,pos.eq,deg,2025-05-13T01:14:58.512Z,GOOD,--,HIPASSJ1145-50_SB72938_F00_B19.ms.tar
0.38012912205564936,visibility,1,WALLABY,ASKAP-72938,scan-642964-3527043,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=scan-642964-3527043,application/x-votable+xml;content=datalink,15749020,WALLABY_1142-53B,177.17178557951806,-51.87909381369503,1.11999,POLYGON ICRS 177.73178057951804 -51.87909381369503 177.73067555730424 -51.84393143671121 177.72736485168065 -51.808907829565186 177.72186152848988 -51.7741612144336 177.71418730683428 -51.739828720332,--,--,--,60781.48490740741,60781.81841435185,28814.99999994412,--,1,0.21484365980537232,0.21556172643644397,--,251,phys.polarisation,/XX/XY/YX/YY/,4,ASKAP,,,em.wl,m,2.8608232313611815e-06,--,--,pos.eq,deg,2025-05-13T01:14:58.512Z,GOOD,--,HIPASSJ1148-50_SB72938_F00_B19.ms.tar
0.38012912205564936,visibility,1,WALLABY,ASKAP-72938,scan-642939-3527018,https://data.csiro.au/casda_vo_proxy/vo/datalink/links?ID=scan-642939-3527018,application/x-votable+xml;content=datalink,13504250,WALLABY_1142-53B,177.17178557951806,-51.87909381369503,1.14011,POLYGON ICRS 177.74184057951805 -51.87909381369503 177.74071570619225 -51.84329976408475 177.73734552557588 -51.80764697723553 177.7317433382322 -51.772276158408864 177.72393125343524 -51.737326900067,--,--,--,60781.48490740741,60781.81841435185,28814.99999994412,--,1,0.20811828182379133,0.2312337824863803,--,144,phys.polarisation,/XX/XY/YX/YY/,4,ASKAP,,,em.wl,m,0.00016052431015686782,--,--,pos.eq,deg,2025-05-13T01:14:58.512Z,GOOD,--,scienceData.WALLABY_1142-53B.SB72938.WALLABY_1142-53B.beam19_averaged_cal.leakage.ms.tar
0.8268966920638349,visibility,1,FLASH,ASKAP-76312,scan-706414-3654177,https://data.csiro.au/casda_vo_

Assuming we have done it correctly, this is a way to get the number of unique observations per source. Lets quickly loop this, and print out some more data as well as appending it to our table.

In [12]:
# Add some columns to our table
solar_like_coords['n_obs'] = [0] * len(solar_like_coords)
solar_like_coords['survey_names'] = Column(['N/A']*len(solar_like_coords), dtype='U100')

In [13]:
# Set some global parameters
min_exptime = 60*60 # 1 Hour minimum time
dataproduct_type = 'visibility' # We want visibility data only

for i, row in enumerate(solar_like_coords):

    # Get the coordinates
    ra = row['Archival_ra']
    dec = row['Archival_dec']
    coords = SkyCoord(ra = ra*u.deg, dec = dec*u.deg, frame='icrs')

    # CASDA Query
    result_table = Casda.query_region(coords, radius="5 arcmin")
    public_data = Casda.filter_out_unreleased(result_table)
    mask = (public_data['dataproduct_type'] == dataproduct_type) & (public_data['t_exptime'] >= min_exptime)
    results = public_data[mask]

    # Here is the data we want to append
    n_obs = len(np.unique(results['obs_id']))
    obs_names = list(np.unique(results['obs_collection']).tolist())

    # Append our data to the table
    solar_like_coords[i]['n_obs'] = n_obs
    solar_like_coords[i]['survey_names'] = ', '.join(obs_names)

In [14]:
total_obs = solar_like_coords['n_obs'].sum()
total_obs

np.int64(81)

In [15]:
solar_like_coords

Simbad_ID,Archival_ra,Archival_dec,n_obs,survey_names
str31,float64,float64,int64,str100
HD 102982,177.787504285457,-51.876071495961,2,"FLASH, WALLABY"
V* V376 Cep,329.577295376914,82.870902021925,0,
V* V815 Her,272.06734740577,29.6910259723,0,
HD 165590A,271.457229,21.44599,0,
HD 82533,142.67769359508,-53.521472152447,3,"EMU, GASKAP-HI"
HD 217344,345.11766870906,-33.745748964489,22,"DINGO, FLASH, WALLABY"
HD 8558,20.839329303534,-57.480907517274,3,"FLASH, WALLABY"
HD 200244,315.742708658278,-22.933914165698,2,FLASH
HD 162186,267.778490987213,-32.307805803789,10,An ASKAP Survey for Galactic Radio Transients in the Baade's Window


In [115]:
solar_like_coords[solar_like_coords['n_obs'] > 0]['Simbad_ID'].tolist()

['HD 102982',
 'HD  82533',
 'HD 217344',
 'HD   8558',
 'HD 200244',
 'HD 162186',
 'BD+21    20A',
 'CPD-55  1885',
 'HD  13183',
 'HD 175897',
 'HD 22213',
 'V* AI Lep',
 'BD+09  4984B',
 'V* TY Col',
 'TYC 9143-577-1',
 'TYC 8927-1888-1',
 'CD-54  2499',
 'ASAS J091509-5700.4',
 'CD-31  8535',
 'CD-48  2972',
 'TYC 8135-2027-1',
 'TYC 7624-1094-1',
 'CD-75   652',
 'HD 205297',
 'HD 202917',
 'HD 220492',
 'HD  52118']

In [116]:
solar_like_coords[solar_like_coords['Simbad_ID'] == 'V* AI Lep']

Simbad_ID,Archival_ra,Archival_dec,n_obs,survey_names
str31,float64,float64,int64,str100
V* AI Lep,85.08647401906606,-19.669755969391588,3,"FLASH, WALLABY"


In [117]:
solar_like_coords[solar_like_coords['Simbad_ID'] == 'V* TY Col']

Simbad_ID,Archival_ra,Archival_dec,n_obs,survey_names
str31,float64,float64,int64,str100
V* TY Col,89.46168456686033,-38.06762489335146,1,FLASH


Above are both of the 'cool' named stars in the sample, they seem like a good starting point since they are more fun to say than HD 59252450248. Let's save our data from this table and move on to actually downloading these visibilities.

In [118]:
solar_like_coords.write('data/source_obs_data.csv', format='csv', overwrite=True)